# 002 · The string and textwrap Modules

> **Standard Library › Text Processing** · Python 3.14

Most string work happens through `str` methods and f-strings, but two small modules fill the gaps: `string` (character-set constants and the simple `$`-substitution `Template`) and `textwrap` (wrapping, shortening, indenting and dedenting blocks of text — something .NET has no built-in API for).

**In this notebook you will learn:**

- the character-class constants in `string` (`ascii_letters`, `digits`, `punctuation`, …)
- `string.Template` for safe, user-editable `$name` templates, and how it compares to composite formatting
- `string.capwords` vs `str.title`
- `textwrap.wrap`, `fill`, `shorten`, `indent` and `dedent` — including the triple-quoted-string `dedent` idiom

## .NET mental model

| Need | .NET | Python |
|---|---|---|
| Character classes | `char.IsLetter`, `char.IsDigit`, … (per char) | `str.isalpha()`, … or the `string` constants |
| Letter/digit alphabets | write them yourself (`"abc..."`) | `string.ascii_letters`, `string.digits`, … |
| Placeholder templates | `string.Format("{0}", x)` / composite formatting | f-strings, `str.format`, or `string.Template` |
| Title-case words | `TextInfo.ToTitleCase` | `str.title()` / `string.capwords()` |
| Word-wrap text | **no BCL API** — hand-rolled loop | `textwrap.wrap` / `textwrap.fill` |
| Truncate with ellipsis | hand-rolled `s[..n] + "…"` | `textwrap.shorten` |
| Indent/dedent lines | raw string literals strip common indentation (C# 11) | `textwrap.indent` / `textwrap.dedent` |

f-strings and `str.format` were covered in *(see Syntax › Data Types › 003 · String Formatting)*; this notebook is about the extras.

## `string` constants

The `string` module exposes ready-made character sets. They are plain `str` objects — handy for validation, generating test data, or building translation tables.

In [ ]:
import string

print(string.ascii_lowercase)
print(string.ascii_uppercase)
print(string.ascii_letters)
print(string.digits, string.hexdigits, string.octdigits)
print(string.punctuation)
print(repr(string.whitespace))
print(len(string.printable))   # digits + letters + punctuation + whitespace

A typical use: strip punctuation from text with `str.translate`, or check a character set.

```csharp
// C#
var clean = new string(text.Where(c => !char.IsPunctuation(c)).ToArray());
```

In [ ]:
text = "Hello, world! (Is this... clean?)"

no_punct = text.translate(str.maketrans("", "", string.punctuation))
print(no_punct)

allowed = set(string.ascii_letters + string.digits + "_")
print(all(ch in allowed for ch in "user_42"), all(ch in allowed for ch in "user-42"))

⚠️ **Gotcha:** the constants are **ASCII only**. `string.ascii_letters` doesn't contain `é` or `ß`, and `string.punctuation` doesn't contain `…` or `—`. For Unicode-aware checks use the `str` methods (`isalpha()`, `isdigit()`, …) or `unicodedata` *(see Standard Library › Text Processing › 004 · Unicode, Bytes and Encodings)*.

In [ ]:
print("é" in string.ascii_letters, "é".isalpha())
print("—" in string.punctuation)

## `string.Template` — `$`-substitution

`string.Template` is a deliberately **simple** template engine: `$name` or `${name}` placeholders, no expressions, no format specs, no attribute access. That simplicity is the point — it's safe to let *end users* or config files supply the template text, because they cannot reach into your objects the way a `str.format` template could (`"{user.__class__}"`).

```csharp
// C# — composite formatting uses positional holes
var msg = string.Format("Hello {0}, your order {1} has shipped", name, orderId);
```

In [ ]:
from string import Template

tpl = Template("Hello $name, your order ${order_id}X has shipped")
print(tpl.substitute(name="alice", order_id=1042))          # keyword arguments
print(tpl.substitute({"name": "bob", "order_id": 7}))        # or a mapping

Use `${name}` when the placeholder is followed by identifier characters (`${order_id}X` above). Write `$$` for a literal dollar sign.

`substitute` raises `KeyError` for a missing value; `safe_substitute` leaves unknown placeholders untouched — useful for multi-stage templating or partial rendering.

In [ ]:
tpl = Template("Dear $name, you owe $$${amount}.")

try:
    tpl.substitute(name="alice")
except KeyError as e:
    print(f"{type(e).__name__}: {e}")

print(tpl.safe_substitute(name="alice"))   # $$ -> $, ${amount} kept as-is
print(tpl.substitute(name="alice", amount="12.50"))

You can inspect a template before rendering it (both methods were added in 3.11):

In [ ]:
tpl = Template("Hi $user, $count new messages")
print(tpl.get_identifiers())   # placeholder names, in order
print(tpl.is_valid())          # False if it contains a malformed $ placeholder
print(Template("cost: $5").is_valid())

**Tip:** which formatting tool when?

| Situation | Use |
|---|---|
| Your own code building a string | f-string |
| Template stored separately, written by developers | `str.format` / `str.format_map` |
| Template written by **users** / translators / config | `string.Template` |

⚠️ **Gotcha:** don't confuse `string.Template` with the **t-strings** introduced in Python 3.14 (`t"Hello {name}"`, which produce a `string.templatelib.Template`). T-strings are f-string-like literals that defer rendering so a library can process the parts (e.g. for safe HTML or SQL); `string.Template` is the old `$`-substitution class.

In [ ]:
from string.templatelib import Template as TString

name = "alice"
t = t"Hello {name}"           # a 3.14 t-string — not rendered yet
print(type(t).__name__, isinstance(t, TString))
print(t.strings, t.values)

## `capwords` vs `title`

`str.title()` capitalises after **every** non-letter, which mangles apostrophes. `string.capwords(s)` splits on whitespace, capitalises each word (lowercasing the rest) and joins with single spaces.

```csharp
// C#
var t = CultureInfo.InvariantCulture.TextInfo.ToTitleCase("it's a dog's life");
```

In [ ]:
s = "it's a   dog's LIFE"
print(s.title())             # "It'S" — oops
print(string.capwords(s))    # note: runs of spaces collapse to one

## `textwrap.wrap` and `textwrap.fill`

`wrap(text, width)` breaks a paragraph into a **list of lines** no longer than `width`; `fill` does the same and joins them with `"\n"`. .NET has no equivalent in the BCL — you'd write the word-splitting loop yourself.

In [ ]:
import textwrap

para = ("Python's standard library ships with a small module for wrapping text, "
        "which is surprisingly useful for console output, emails and docstrings.")

for line in textwrap.wrap(para, width=40):
    print(f"|{line:<40}|")

In [ ]:
print(textwrap.fill(para, width=50, initial_indent="* ", subsequent_indent="  "))

Other useful keyword arguments include `max_lines=` (with `placeholder=`), `break_long_words=`, and `break_on_hyphens=`. Whitespace (including newlines) inside the input is collapsed, so wrap **one paragraph at a time**.

In [ ]:
long_word = "Supercalifragilisticexpialidocious is long"
print(textwrap.wrap(long_word, width=12))
print(textwrap.wrap(long_word, width=12, break_long_words=False))
print(textwrap.fill(para, width=30, max_lines=2, placeholder=" [...]"))

## `textwrap.shorten`

`shorten` collapses whitespace and truncates **at a word boundary** so the result fits in `width`, appending a placeholder (default `" [...]"`). Great for previews and log messages.

```csharp
// C# — the usual hand-rolled version cuts mid-word
var preview = s.Length > 25 ? s[..22] + "..." : s;
```

In [ ]:
title = "The quick   brown fox jumps over the lazy dog"
print(textwrap.shorten(title, width=25))
print(textwrap.shorten(title, width=25, placeholder="…"))
print(textwrap.shorten("short", width=25))   # unchanged (apart from whitespace)

## `textwrap.indent`

`indent(text, prefix)` adds a prefix to every line — perfect for quoting or nesting output. Blank lines are skipped by default; pass a `predicate` to control which lines get the prefix.

In [ ]:
body = "first line\n\nthird line"
print(textwrap.indent(body, "> "))
print(textwrap.indent(body, "> ", predicate=lambda line: True))   # prefix blank lines too

## `textwrap.dedent` — tidy triple-quoted strings

Triple-quoted strings keep the source indentation, which is ugly when the literal lives inside a function. `dedent` removes the **common leading whitespace** from all lines.

C# 11 raw string literals do this automatically based on the closing `"""`; Python doesn't, so `dedent` is the idiom:

```csharp
// C# 11 — indentation before the closing """ is removed
var sql = """
    SELECT *
    FROM orders
    """;
```

In [ ]:
def build_query() -> str:
    return textwrap.dedent("""\
        SELECT id, total
        FROM orders
        WHERE status = 'shipped'
    """)

print(build_query())

**Tip:** the backslash right after the opening `"""` suppresses the first newline. Without it the string starts with an empty line. Combine with `.strip()` if you also want to drop the trailing newline.

⚠️ **Gotcha:** `dedent` only removes whitespace that is common to *every* non-blank line, and it treats tabs and spaces as different characters — mixing them means nothing gets removed.

In [ ]:
mixed = "    four spaces\n\ttab"
print(repr(textwrap.dedent(mixed)))   # nothing in common -> unchanged

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| ASCII letters / digits | hand-written string | `string.ascii_letters`, `string.digits` |
| Punctuation set | `char.IsPunctuation` | `string.punctuation` (ASCII only) |
| User-supplied template | composite formatting (unsafe-ish) | `string.Template(...).substitute(...)` |
| Partial substitution | — | `Template.safe_substitute` |
| Title-case words | `TextInfo.ToTitleCase` | `string.capwords(s)` |
| Word-wrap to lines | hand-rolled | `textwrap.wrap(s, width)` |
| Word-wrap to one string | hand-rolled | `textwrap.fill(s, width)` |
| Truncate at word boundary | hand-rolled | `textwrap.shorten(s, width)` |
| Prefix every line | `string.Join` + `Split` | `textwrap.indent(s, prefix)` |
| Strip common indentation | raw string literal `"""` (C# 11) | `textwrap.dedent(s)` |

## Exercises

1. Write `random_code(n)` that returns an `n`-character code made of uppercase ASCII letters and digits, using `string` constants and `random.Random(42)` for reproducibility.
2. Given the template text `"Hi $user, your plan $plan renews on $date"`, render it with `user="alice"` and `plan="Pro"` only, leaving `$date` in place.
3. Format a docstring-like triple-quoted string inside a function so it has no leading indentation, then re-indent it with four spaces and wrap it to 30 columns.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import random
import string
import textwrap
from string import Template

# 1
rng = random.Random(42)

def random_code(n: int) -> str:
    alphabet = string.ascii_uppercase + string.digits
    return "".join(rng.choice(alphabet) for _ in range(n))

print(random_code(8))

# 2 — safe_substitute leaves unknown placeholders alone
tpl = Template("Hi $user, your plan $plan renews on $date")
print(tpl.safe_substitute(user="alice", plan="Pro"))

# 3 — dedent, wrap the paragraph, then indent
def describe() -> str:
    text = textwrap.dedent("""\
        This function does something remarkably useful
        and its description is rather long.
    """)
    return textwrap.indent(textwrap.fill(text, width=30), "    ")

print(describe())
```

</details>